# 04 — openFDA Drug Labels Exploration

**Stage 5 of the pipeline:** add **FDA drug-label / indication evidence** for each EGFR drug.

Flow: load drug recommendations (notebook 01) → search openFDA labels per drug → extract names, indications, warnings, and whether the label mentions EGFR → save evidence + per-drug summary.

Outputs: `egfr_openfda_labels.csv`, `egfr_openfda_summary.csv`

### 1. Test notebook environment

In [1]:
import sys
import time
from pathlib import Path

import requests
import pandas as pd

print("Notebook is working")
print("Python executable:", sys.executable)

Notebook is working
Python executable: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/.venv/bin/python


### 2. Set project folders

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)

Project root: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant
Processed data folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed


### 3. Load EGFR drug recommendations (from notebook 01)

In [3]:
recommendations_file = PROCESSED_DIR / "egfr_drug_recommendations.csv"

if not recommendations_file.exists():
    raise FileNotFoundError(
        "egfr_drug_recommendations.csv not found. Run Notebook 01 first."
    )

drug_recommendations_df = pd.read_csv(recommendations_file)
print("Rows:", len(drug_recommendations_df))
drug_recommendations_df.head()

Rows: 76


,drug_name,molecule_chembl_id,action_type,mechanism_of_action,approval_status,max_phase,target_name
0,PANITUMUMAB,CHEMBL1201827,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor
1,CETUXIMAB,CHEMBL1201577,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor
2,ERLOTINIB HYDROCHLORIDE,CHEMBL1079742,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor
3,GEFITINIB,CHEMBL939,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor
4,LAPATINIB DITOSYLATE,CHEMBL1201179,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor


### 4. Select drugs to search
We prefer approved drugs (they have official labels). If none are tagged approved, we fall back to the first 10 drugs.

In [4]:
target_name = "EGFR"

if "drug_name" not in drug_recommendations_df.columns:
    raise ValueError("drug_name column not found in egfr_drug_recommendations.csv")

if "approval_status" in drug_recommendations_df.columns:
    approved_drugs_df = drug_recommendations_df[
        drug_recommendations_df["approval_status"].str.contains("Approved", case=False, na=False)
    ].copy()
else:
    approved_drugs_df = drug_recommendations_df.copy()

if approved_drugs_df.empty:
    print("No drugs tagged 'Approved' -> using first 10 drugs instead.")
    approved_drugs_df = drug_recommendations_df.copy()

drugs_to_search = (
    approved_drugs_df["drug_name"].dropna().drop_duplicates().head(10).tolist()
)

print("Target:", target_name)
print("Drugs selected for openFDA label search:")
for drug in drugs_to_search:
    print("-", drug)

Target: EGFR
Drugs selected for openFDA label search:
- PANITUMUMAB
- CETUXIMAB
- ERLOTINIB HYDROCHLORIDE
- GEFITINIB
- LAPATINIB DITOSYLATE
- AFATINIB DIMALEATE
- OSIMERTINIB MESYLATE
- NECITUMUMAB
- OLMUTINIB
- BRIGATINIB


### 5. openFDA helper (404 = no records, not an error)

In [5]:
OPENFDA_LABEL_URL = "https://api.fda.gov/drug/label.json"


def openfda_get(params, retries=3, pause=1):
    """GET JSON from openFDA. A 404 means 'no matching records' -> return empty."""
    for attempt in range(retries):
        try:
            response = requests.get(OPENFDA_LABEL_URL, params=params, timeout=(10, 60))
            if response.status_code == 200:
                return response.json()
            if response.status_code == 404:
                return {"results": []}
            print(f"  attempt {attempt + 1}: HTTP {response.status_code}, retrying...")
        except requests.exceptions.RequestException as error:
            print(f"  attempt {attempt + 1}: {type(error).__name__}, retrying...")
        time.sleep(pause)
    return {"results": []}

### 6. Helpers for queries and field extraction
Note: drug names from ChEMBL may include a salt suffix (e.g. *OSIMERTINIB MESYLATE*). We also try the **base name** (first word) so more drugs match their FDA label.

In [6]:
def build_openfda_queries(drug_name):
    """Try exact name, brand name, base name (salt stripped), then broad text."""
    name = str(drug_name).strip()
    base = name.split()[0] if name else name
    queries = [
        f'openfda.generic_name:"{name}"',
        f'openfda.brand_name:"{name}"',
    ]
    if base and base != name:
        queries += [
            f'openfda.generic_name:"{base}"',
            f'openfda.brand_name:"{base}"',
        ]
    queries.append(f'"{name}"')
    return queries


def safe_first(value):
    if isinstance(value, list):
        return value[0] if value else None
    return value


def safe_join(value, max_chars=800):
    if isinstance(value, list):
        text = " ".join(str(item) for item in value)
    elif value is None:
        text = ""
    else:
        text = str(value)
    text = " ".join(text.split())
    return text[:max_chars] + "..." if len(text) > max_chars else text


def get_openfda_field(record, field_name):
    return safe_first(record.get("openfda", {}).get(field_name))


def label_mentions_target(record, target_name="EGFR"):
    parts = [
        safe_join(record.get("indications_and_usage"), 5000),
        safe_join(record.get("clinical_studies"), 5000),
        safe_join(record.get("mechanism_of_action"), 5000),
        safe_join(record.get("description"), 5000),
    ]
    full = " ".join(parts).lower()
    terms = [target_name.lower(), "epidermal growth factor receptor"]
    return any(t in full for t in terms)


def extract_label_record(record, drug_name, query, target_name="EGFR"):
    set_id = record.get("set_id")
    return {
        "target_name": target_name,
        "drug_name": drug_name,
        "query": query,
        "set_id": set_id,
        "brand_name": get_openfda_field(record, "brand_name"),
        "generic_name": get_openfda_field(record, "generic_name"),
        "manufacturer_name": get_openfda_field(record, "manufacturer_name"),
        "route": get_openfda_field(record, "route"),
        "indications_and_usage": safe_join(record.get("indications_and_usage")),
        "mechanism_of_action": safe_join(record.get("mechanism_of_action")),
        "warnings": safe_join(record.get("warnings"), 500),
        "mentions_target_in_label": label_mentions_target(record, target_name),
        "label_url": f"https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid={set_id}" if set_id else None,
        "source": "openFDA drug label",
    }

### 7. Quick test for one drug

In [7]:
test_drug = drugs_to_search[0]
test_queries = build_openfda_queries(test_drug)
print("Test drug:", test_drug)

test_data, test_query_used = None, None
for query in test_queries:
    data = openfda_get({"search": query, "limit": 3})
    if data.get("results"):
        test_data, test_query_used = data, query
        break
    time.sleep(0.3)

if test_data is None:
    print("No openFDA label found for test drug.")
else:
    print("Query used:", test_query_used)
    print("Records found:", len(test_data.get("results", [])))

Test drug: PANITUMUMAB
Query used: "PANITUMUMAB"
Records found: 3


### 8. Search openFDA labels for all selected drugs

In [8]:
openfda_label_records = []

for drug in drugs_to_search:
    found_results, query_used = [], None
    for query in build_openfda_queries(drug):
        data = openfda_get({"search": query, "limit": 3})
        results = data.get("results", [])
        if results:
            found_results, query_used = results, query
            break
        time.sleep(0.3)

    print(f"{drug}: found {len(found_results)} label records")
    for record in found_results:
        openfda_label_records.append(
            extract_label_record(record, drug, query_used, target_name)
        )
    time.sleep(0.4)

openfda_labels_df = pd.DataFrame(openfda_label_records)
print("Total openFDA label records collected:", len(openfda_labels_df))
openfda_labels_df.head(10)

PANITUMUMAB: found 3 label records
CETUXIMAB: found 1 label records
ERLOTINIB HYDROCHLORIDE: found 3 label records
GEFITINIB: found 3 label records
LAPATINIB DITOSYLATE: found 3 label records
AFATINIB DIMALEATE: found 1 label records
OSIMERTINIB MESYLATE: found 1 label records
NECITUMUMAB: found 1 label records
OLMUTINIB: found 0 label records
BRIGATINIB: found 1 label records
Total openFDA label records collected: 17


,target_name,drug_name,query,set_id,brand_name,generic_name,manufacturer_name,route,indications_and_usage,mechanism_of_action,warnings,mentions_target_in_label,label_url,source
0,EGFR,PANITUMUMAB,"""PANITUMUMAB""",824f19c9-0546-4a8a-8d8f-c4055c04f7c7,Stivarga,REGORAFENIB,Bayer HealthCare Pharmaceuticals Inc.,ORAL,1 INDICATIONS AND USAGE STIVARGA is a kinase i...,12.1 Mechanism of Action Regorafenib is a smal...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
1,EGFR,PANITUMUMAB,"""PANITUMUMAB""",c80a362c-7ac3-4894-a076-0691e68ef8c1,LUMAKRAS,SOTORASIB,Amgen Inc,ORAL,1 INDICATIONS AND USAGE LUMAKRAS is an inhibit...,12.1 Mechanism of Action Sotorasib is an inhib...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
2,EGFR,PANITUMUMAB,"""PANITUMUMAB""",e0fa4bca-f245-4d92-ae29-b0c630a315c2,NaN,NaN,NaN,NaN,1 INDICATIONS AND USAGE Vectibix is an epiderm...,12.1 Mechanism of Action The EGFR is a transme...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
3,EGFR,CETUXIMAB,"openfda.generic_name:""CETUXIMAB""",8bc6397e-4bd8-4d37-a007-a327e4da34d9,ERBITUX,CETUXIMAB,ImClone LLC,INTRAVENOUS,1 INDICATIONS AND USAGE ERBITUX ® is an epider...,12.1 Mechanism of Action The epidermal growth ...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
4,EGFR,ERLOTINIB HYDROCHLORIDE,"openfda.generic_name:""ERLOTINIB HYDROCHLORIDE""",2c4c7353-b81c-4651-8aac-6d15e88145af,Erlotinib,ERLOTINIB HYDROCHLORIDE,Armas Pharmaceuticals Inc.,ORAL,1 INDICATIONS AND USAGE Erlotinib tablet is a ...,12.1 Mechanism of Action Epidermal growth fact...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
5,EGFR,ERLOTINIB HYDROCHLORIDE,"openfda.generic_name:""ERLOTINIB HYDROCHLORIDE""",4cff04f5-3693-4d87-a9f8-a2b195e3d26e,Erlotinib Hydrochloride,ERLOTINIB HYDROCHLORIDE,Novadoz Pharmaceuticals LLC,ORAL,1 INDICATIONS AND USAGE Erlotinib tablet is a ...,12.1 Mechanism of Action Epidermal growth fact...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
6,EGFR,ERLOTINIB HYDROCHLORIDE,"openfda.generic_name:""ERLOTINIB HYDROCHLORIDE""",53cf8868-cfd8-4c6e-ad9d-3171ff1e4d8c,Erlotinib,ERLOTINIB HYDROCHLORIDE,Armas Pharmaceuticals Inc.,ORAL,1 INDICATIONS AND USAGE Erlotinib tablet is a ...,12.1 Mechanism of Action Epidermal growth fact...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
7,EGFR,GEFITINIB,"openfda.generic_name:""GEFITINIB""",0afc12dd-12f8-4def-9027-f38412a862b9,Gefitinib,GEFITINIB,"Qilu Pharmaceutical Co., Ltd.",ORAL,1 INDICATIONS AND USAGE Gefitinib tablets are ...,12.1 Mechanism of Action The epidermal growth ...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
8,EGFR,GEFITINIB,"openfda.generic_name:""GEFITINIB""",1a3c0ce7-06a1-4e04-9106-14ddb2a866a5,Gefitinib,GEFITINIB,Natco Pharma USA LLC,ORAL,1 INDICATIONS AND USAGE Gefitinib tablets are ...,12.1 Mechanism of Action The epidermal growth ...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
9,EGFR,GEFITINIB,"openfda.generic_name:""GEFITINIB""",3dd135f0-5db1-4236-9756-04533b66dc9d,Gefitinib,GEFITINIB,"Teva Pharmaceuticals, Inc.",ORAL,1 INDICATIONS AND USAGE Gefitinib tablets are ...,12.1 Mechanism of Action The epidermal growth ...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label


### 9. Remove duplicate labels (same drug + set_id)

In [9]:
if openfda_labels_df.empty:
    clean_openfda_labels_df = pd.DataFrame(
        columns=["target_name", "drug_name", "set_id", "brand_name", "generic_name",
                 "indications_and_usage", "mechanism_of_action",
                 "mentions_target_in_label", "label_url", "source"]
    )
else:
    clean_openfda_labels_df = openfda_labels_df.drop_duplicates(
        subset=["drug_name", "set_id"]
    ).copy()

print("Rows before cleaning:", len(openfda_labels_df))
print("Rows after cleaning:", len(clean_openfda_labels_df))
clean_openfda_labels_df.head(10)

Rows before cleaning: 15
Rows after cleaning: 15


,target_name,drug_name,query,set_id,brand_name,generic_name,manufacturer_name,route,indications_and_usage,mechanism_of_action,warnings,mentions_target_in_label,label_url,source
0,EGFR,PANITUMUMAB,"openfda.generic_name:""PANITUMUMAB""",e0fa4bca-f245-4d92-ae29-b0c630a315c2,Vectibix,PANITUMUMAB,"Amgen, Inc",INTRAVENOUS,1 INDICATIONS AND USAGE Vectibix is an epiderm...,12.1 Mechanism of Action The EGFR is a transme...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
1,EGFR,CETUXIMAB,"openfda.generic_name:""CETUXIMAB""",8bc6397e-4bd8-4d37-a007-a327e4da34d9,ERBITUX,CETUXIMAB,ImClone LLC,INTRAVENOUS,1 INDICATIONS AND USAGE ERBITUX ® is an epider...,12.1 Mechanism of Action The epidermal growth ...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
2,EGFR,ERLOTINIB HYDROCHLORIDE,"openfda.generic_name:""ERLOTINIB HYDROCHLORIDE""",2c4c7353-b81c-4651-8aac-6d15e88145af,Erlotinib,ERLOTINIB HYDROCHLORIDE,Armas Pharmaceuticals Inc.,ORAL,1 INDICATIONS AND USAGE Erlotinib tablet is a ...,12.1 Mechanism of Action Epidermal growth fact...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
3,EGFR,ERLOTINIB HYDROCHLORIDE,"openfda.generic_name:""ERLOTINIB HYDROCHLORIDE""",4cff04f5-3693-4d87-a9f8-a2b195e3d26e,Erlotinib Hydrochloride,ERLOTINIB HYDROCHLORIDE,Novadoz Pharmaceuticals LLC,ORAL,1 INDICATIONS AND USAGE Erlotinib tablet is a ...,12.1 Mechanism of Action Epidermal growth fact...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
4,EGFR,ERLOTINIB HYDROCHLORIDE,"openfda.generic_name:""ERLOTINIB HYDROCHLORIDE""",53cf8868-cfd8-4c6e-ad9d-3171ff1e4d8c,Erlotinib,ERLOTINIB HYDROCHLORIDE,Armas Pharmaceuticals Inc.,ORAL,1 INDICATIONS AND USAGE Erlotinib tablet is a ...,12.1 Mechanism of Action Epidermal growth fact...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
5,EGFR,GEFITINIB,"openfda.generic_name:""GEFITINIB""",0afc12dd-12f8-4def-9027-f38412a862b9,Gefitinib,GEFITINIB,"Qilu Pharmaceutical Co., Ltd.",ORAL,1 INDICATIONS AND USAGE Gefitinib tablets are ...,12.1 Mechanism of Action The epidermal growth ...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
6,EGFR,GEFITINIB,"openfda.generic_name:""GEFITINIB""",1a3c0ce7-06a1-4e04-9106-14ddb2a866a5,Gefitinib,GEFITINIB,Natco Pharma USA LLC,ORAL,1 INDICATIONS AND USAGE Gefitinib tablets are ...,12.1 Mechanism of Action The epidermal growth ...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
7,EGFR,GEFITINIB,"openfda.generic_name:""GEFITINIB""",3dd135f0-5db1-4236-9756-04533b66dc9d,Gefitinib,GEFITINIB,"Teva Pharmaceuticals, Inc.",ORAL,1 INDICATIONS AND USAGE Gefitinib tablets are ...,12.1 Mechanism of Action The epidermal growth ...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
8,EGFR,LAPATINIB DITOSYLATE,"openfda.generic_name:""LAPATINIB""",302a4346-a68d-af3e-e063-6294a90a4a6f,Lapatinib,LAPATINIB,AvKARE,ORAL,1 INDICATIONS AND USAGE Lapatinib tablets are ...,12.1 Mechanism of Action Lapatinib is a 4-anil...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
9,EGFR,LAPATINIB DITOSYLATE,"openfda.generic_name:""LAPATINIB""",50d0dd0c-7682-43d0-aa1b-ca26e9181aee,Lapatinib,LAPATINIB,"Lupin Pharmaceuticals, Inc.",ORAL,1 INDICATIONS AND USAGE Lapatinib tablets are ...,12.1 Mechanism of Action Lapatinib is a 4-anil...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label


### 10. Save openFDA label evidence dataset

In [10]:
openfda_labels_file = PROCESSED_DIR / "egfr_openfda_labels.csv"
clean_openfda_labels_df.to_csv(openfda_labels_file, index=False)
print("Saved:", openfda_labels_file)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_openfda_labels.csv


### 11. Per-drug openFDA summary

In [11]:
if clean_openfda_labels_df.empty:
    openfda_summary_df = pd.DataFrame(
        columns=["target_name", "drug_name", "openfda_label_count", "has_openfda_label",
                 "mentions_target_in_label", "brand_names", "generic_names", "top_indications"]
    )
else:
    openfda_summary_df = (
        clean_openfda_labels_df.groupby(["target_name", "drug_name"])
        .agg(
            openfda_label_count=("set_id", "nunique"),
            mentions_target_in_label=("mentions_target_in_label", "max"),
            brand_names=("brand_name", lambda s: " | ".join(list(s.dropna().drop_duplicates())[:5])),
            generic_names=("generic_name", lambda s: " | ".join(list(s.dropna().drop_duplicates())[:5])),
            top_indications=("indications_and_usage", lambda s: " | ".join(list(s.dropna())[:2])),
        )
        .reset_index()
    )
    openfda_summary_df["has_openfda_label"] = openfda_summary_df["openfda_label_count"] > 0

openfda_summary_df

,target_name,drug_name,openfda_label_count,mentions_target_in_label,brand_names,generic_names,top_indications,has_openfda_label
0,EGFR,AFATINIB DIMALEATE,1,True,Gilotrif,AFATINIB,1 INDICATIONS AND USAGE GILOTRIF is a kinase i...,True
1,EGFR,BRIGATINIB,1,True,Alunbrig,BRIGATINIB,1 INDICATIONS AND USAGE ALUNBRIG is indicated ...,True
2,EGFR,CETUXIMAB,1,True,ERBITUX,CETUXIMAB,1 INDICATIONS AND USAGE ERBITUX ® is an epider...,True
3,EGFR,ERLOTINIB HYDROCHLORIDE,3,True,Erlotinib | Erlotinib Hydrochloride,ERLOTINIB HYDROCHLORIDE,1 INDICATIONS AND USAGE Erlotinib tablet is a ...,True
4,EGFR,GEFITINIB,3,True,Gefitinib,GEFITINIB,1 INDICATIONS AND USAGE Gefitinib tablets are ...,True
5,EGFR,LAPATINIB DITOSYLATE,3,True,Lapatinib,LAPATINIB,1 INDICATIONS AND USAGE Lapatinib tablets are ...,True
6,EGFR,NECITUMUMAB,1,True,,,1 INDICATIONS AND USAGE PORTRAZZA™ is an epide...,True
7,EGFR,OSIMERTINIB MESYLATE,1,True,TAGRISSO,OSIMERTINIB,1 INDICATIONS AND USAGE TAGRISSO is a kinase i...,True
8,EGFR,PANITUMUMAB,1,True,Vectibix,PANITUMUMAB,1 INDICATIONS AND USAGE Vectibix is an epiderm...,True


### 12. Add an openFDA evidence score

In [12]:
def calculate_openfda_evidence_score(row):
    score = 0.0
    if row.get("has_openfda_label") is True:
        score += 0.5
    if row.get("mentions_target_in_label") is True:
        score += 0.3
    if pd.notna(row.get("top_indications")) and str(row.get("top_indications")).strip():
        score += 0.2
    return round(min(score, 1.0), 2)


if not openfda_summary_df.empty:
    openfda_summary_df["openfda_evidence_score"] = openfda_summary_df.apply(
        calculate_openfda_evidence_score, axis=1
    )
else:
    openfda_summary_df["openfda_evidence_score"] = []

openfda_summary_df

,target_name,drug_name,openfda_label_count,mentions_target_in_label,brand_names,generic_names,top_indications,has_openfda_label,openfda_evidence_score
0,EGFR,AFATINIB DIMALEATE,1,True,Gilotrif,AFATINIB,1 INDICATIONS AND USAGE GILOTRIF is a kinase i...,True,1.0
1,EGFR,BRIGATINIB,1,True,Alunbrig,BRIGATINIB,1 INDICATIONS AND USAGE ALUNBRIG is indicated ...,True,1.0
2,EGFR,CETUXIMAB,1,True,ERBITUX,CETUXIMAB,1 INDICATIONS AND USAGE ERBITUX ® is an epider...,True,1.0
3,EGFR,ERLOTINIB HYDROCHLORIDE,3,True,Erlotinib | Erlotinib Hydrochloride,ERLOTINIB HYDROCHLORIDE,1 INDICATIONS AND USAGE Erlotinib tablet is a ...,True,1.0
4,EGFR,GEFITINIB,3,True,Gefitinib,GEFITINIB,1 INDICATIONS AND USAGE Gefitinib tablets are ...,True,1.0
5,EGFR,LAPATINIB DITOSYLATE,3,True,Lapatinib,LAPATINIB,1 INDICATIONS AND USAGE Lapatinib tablets are ...,True,1.0
6,EGFR,NECITUMUMAB,1,True,,,1 INDICATIONS AND USAGE PORTRAZZA™ is an epide...,True,1.0
7,EGFR,OSIMERTINIB MESYLATE,1,True,TAGRISSO,OSIMERTINIB,1 INDICATIONS AND USAGE TAGRISSO is a kinase i...,True,1.0
8,EGFR,PANITUMUMAB,1,True,Vectibix,PANITUMUMAB,1 INDICATIONS AND USAGE Vectibix is an epiderm...,True,1.0


### 13. Save openFDA summary dataset

In [13]:
openfda_summary_file = PROCESSED_DIR / "egfr_openfda_summary.csv"
openfda_summary_df.to_csv(openfda_summary_file, index=False)
print("Saved:", openfda_summary_file)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_openfda_summary.csv


### 14. Final result

In [14]:
print("openFDA Drug Label Exploration Complete")
print("=" * 70)
print("Target:", target_name)
print("Drugs searched:", len(drugs_to_search))
print("openFDA label records:", len(clean_openfda_labels_df))
print("openFDA summary rows:", len(openfda_summary_df))
display(openfda_summary_df)
display(clean_openfda_labels_df.head(10))

openFDA Drug Label Exploration Complete
Target: EGFR
Drugs searched: 10
openFDA label records: 15
openFDA summary rows: 9


,target_name,drug_name,openfda_label_count,mentions_target_in_label,brand_names,generic_names,top_indications,has_openfda_label,openfda_evidence_score
0,EGFR,AFATINIB DIMALEATE,1,True,Gilotrif,AFATINIB,1 INDICATIONS AND USAGE GILOTRIF is a kinase i...,True,1.0
1,EGFR,BRIGATINIB,1,True,Alunbrig,BRIGATINIB,1 INDICATIONS AND USAGE ALUNBRIG is indicated ...,True,1.0
2,EGFR,CETUXIMAB,1,True,ERBITUX,CETUXIMAB,1 INDICATIONS AND USAGE ERBITUX ® is an epider...,True,1.0
3,EGFR,ERLOTINIB HYDROCHLORIDE,3,True,Erlotinib | Erlotinib Hydrochloride,ERLOTINIB HYDROCHLORIDE,1 INDICATIONS AND USAGE Erlotinib tablet is a ...,True,1.0
4,EGFR,GEFITINIB,3,True,Gefitinib,GEFITINIB,1 INDICATIONS AND USAGE Gefitinib tablets are ...,True,1.0
5,EGFR,LAPATINIB DITOSYLATE,3,True,Lapatinib,LAPATINIB,1 INDICATIONS AND USAGE Lapatinib tablets are ...,True,1.0
6,EGFR,NECITUMUMAB,1,True,,,1 INDICATIONS AND USAGE PORTRAZZA™ is an epide...,True,1.0
7,EGFR,OSIMERTINIB MESYLATE,1,True,TAGRISSO,OSIMERTINIB,1 INDICATIONS AND USAGE TAGRISSO is a kinase i...,True,1.0
8,EGFR,PANITUMUMAB,1,True,Vectibix,PANITUMUMAB,1 INDICATIONS AND USAGE Vectibix is an epiderm...,True,1.0


,target_name,drug_name,query,set_id,brand_name,generic_name,manufacturer_name,route,indications_and_usage,mechanism_of_action,warnings,mentions_target_in_label,label_url,source
0,EGFR,PANITUMUMAB,"openfda.generic_name:""PANITUMUMAB""",e0fa4bca-f245-4d92-ae29-b0c630a315c2,Vectibix,PANITUMUMAB,"Amgen, Inc",INTRAVENOUS,1 INDICATIONS AND USAGE Vectibix is an epiderm...,12.1 Mechanism of Action The EGFR is a transme...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
1,EGFR,CETUXIMAB,"openfda.generic_name:""CETUXIMAB""",8bc6397e-4bd8-4d37-a007-a327e4da34d9,ERBITUX,CETUXIMAB,ImClone LLC,INTRAVENOUS,1 INDICATIONS AND USAGE ERBITUX ® is an epider...,12.1 Mechanism of Action The epidermal growth ...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
2,EGFR,ERLOTINIB HYDROCHLORIDE,"openfda.generic_name:""ERLOTINIB HYDROCHLORIDE""",2c4c7353-b81c-4651-8aac-6d15e88145af,Erlotinib,ERLOTINIB HYDROCHLORIDE,Armas Pharmaceuticals Inc.,ORAL,1 INDICATIONS AND USAGE Erlotinib tablet is a ...,12.1 Mechanism of Action Epidermal growth fact...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
3,EGFR,ERLOTINIB HYDROCHLORIDE,"openfda.generic_name:""ERLOTINIB HYDROCHLORIDE""",4cff04f5-3693-4d87-a9f8-a2b195e3d26e,Erlotinib Hydrochloride,ERLOTINIB HYDROCHLORIDE,Novadoz Pharmaceuticals LLC,ORAL,1 INDICATIONS AND USAGE Erlotinib tablet is a ...,12.1 Mechanism of Action Epidermal growth fact...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
4,EGFR,ERLOTINIB HYDROCHLORIDE,"openfda.generic_name:""ERLOTINIB HYDROCHLORIDE""",53cf8868-cfd8-4c6e-ad9d-3171ff1e4d8c,Erlotinib,ERLOTINIB HYDROCHLORIDE,Armas Pharmaceuticals Inc.,ORAL,1 INDICATIONS AND USAGE Erlotinib tablet is a ...,12.1 Mechanism of Action Epidermal growth fact...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
5,EGFR,GEFITINIB,"openfda.generic_name:""GEFITINIB""",0afc12dd-12f8-4def-9027-f38412a862b9,Gefitinib,GEFITINIB,"Qilu Pharmaceutical Co., Ltd.",ORAL,1 INDICATIONS AND USAGE Gefitinib tablets are ...,12.1 Mechanism of Action The epidermal growth ...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
6,EGFR,GEFITINIB,"openfda.generic_name:""GEFITINIB""",1a3c0ce7-06a1-4e04-9106-14ddb2a866a5,Gefitinib,GEFITINIB,Natco Pharma USA LLC,ORAL,1 INDICATIONS AND USAGE Gefitinib tablets are ...,12.1 Mechanism of Action The epidermal growth ...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
7,EGFR,GEFITINIB,"openfda.generic_name:""GEFITINIB""",3dd135f0-5db1-4236-9756-04533b66dc9d,Gefitinib,GEFITINIB,"Teva Pharmaceuticals, Inc.",ORAL,1 INDICATIONS AND USAGE Gefitinib tablets are ...,12.1 Mechanism of Action The epidermal growth ...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
8,EGFR,LAPATINIB DITOSYLATE,"openfda.generic_name:""LAPATINIB""",302a4346-a68d-af3e-e063-6294a90a4a6f,Lapatinib,LAPATINIB,AvKARE,ORAL,1 INDICATIONS AND USAGE Lapatinib tablets are ...,12.1 Mechanism of Action Lapatinib is a 4-anil...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
9,EGFR,LAPATINIB DITOSYLATE,"openfda.generic_name:""LAPATINIB""",50d0dd0c-7682-43d0-aa1b-ca26e9181aee,Lapatinib,LAPATINIB,"Lupin Pharmaceuticals, Inc.",ORAL,1 INDICATIONS AND USAGE Lapatinib tablets are ...,12.1 Mechanism of Action Lapatinib is a 4-anil...,,True,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,openFDA drug label
